# Car Price Prediction Using Regression
> **Task 3 - Horizon TechX Internship**

**Student:** Tharani Natarajan
**College:** IFET College of Engineering
**Department:** Artificial Intelligence and Data Science

## 1. Project Introduction
In this project, we build a professional machine learning pipeline to predict the selling price of a car using various vehicle features. The dataset is sourced from the UCI Machine Learning Repository.

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

## 3. Dataset Source
**Source:** UCI Automobile Data Set (1985)
*Note: This public dataset is used independently for this task and was not provided by Horizon TechX.*

## 4. Load Dataset

In [ ]:
df = pd.read_csv('../data/car_price_dataset.csv')
df.head()

## 5. Dataset Inspection

In [ ]:
print('Shape:', df.shape)
print('Data Types:')
print(df.dtypes)
print('Missing Values:')
print(df.isnull().sum())

## 6. Data Cleaning
We will drop rows where the target ('price') is missing and ensure proper numeric types.

In [ ]:
df = df.dropna(subset=['price'])
cols_to_numeric = ['normalized-losses', 'bore', 'stroke', 'horsepower', 'peak-rpm']
for col in cols_to_numeric:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
if df.duplicated().sum() > 0:
    df = df.drop_duplicates()
print('Cleaned Shape:', df.shape)

## 7. Exploratory Data Analysis

In [ ]:
sns.set_theme(style='whitegrid')
plt.figure(figsize=(8,5))
sns.histplot(df['price'], kde=True, color='blue')
plt.title('Distribution of Car Prices')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x='horsepower', y='price', data=df, hue='fuel-type', alpha=0.7)
plt.title('Car Price vs Horsepower')
plt.show()

## 8. Feature Engineering
Creating a combined MPG feature (`avg-mpg`) from city and highway MPG.

In [ ]:
df['avg-mpg'] = (df['city-mpg'] + df['highway-mpg']) / 2.0
df.head()

## 9. Train-Test Split

In [ ]:
numerical_features = ['wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-size', 
                      'bore', 'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'avg-mpg']
categorical_features = ['make', 'fuel-type', 'aspiration', 'num-of-doors', 'body-style', 
                        'drive-wheels', 'engine-location', 'engine-type', 'num-of-cylinders', 'fuel-system']

X = df[numerical_features + categorical_features]
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

## 10. Preprocessing Pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

## 11. Baseline Model
We use Linear Regression as the baseline.

In [ ]:
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', LinearRegression())])
lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)
print('Baseline R2:', r2_score(y_test, lr_preds))

## 12. Improved Models & Comparison
Training Random Forest and Gradient Boosting regressors.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': Gradient BoostingRegressor(n_estimators=100, random_state=42)
}

results = []
best_r2 = -1
best_model = None
best_preds = None

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    r2 = r2_score(y_test, preds)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
        'R2 Score': r2
    })
    if r2 > best_r2:
        best_r2 = r2
        best_model = pipeline
        best_preds = preds

results_df = pd.DataFrame(results)
results_df

## 13. Actual vs Predicted Analysis

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(y_test, best_preds, alpha=0.7, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title('Actual vs Predicted Prices')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.show()

## 14. Feature Importance
Extracting feature importances from the best tree-based model.

In [ ]:
if hasattr(best_model.named_steps['regressor'], 'feature_importances_'):
    cat_encoder = best_model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
    cat_features_out = cat_encoder.get_feature_names_out(categorical_features)
    all_features = numerical_features + list(cat_features_out)
    importances = best_model.named_steps['regressor'].feature_importances_
    indices = np.argsort(importances)[-10:]
    
    plt.figure(figsize=(8,5))
    plt.barh(range(len(indices)), importances[indices], color='c')
    plt.yticks(range(len(indices)), [all_features[i] for i in indices])
    plt.title('Top 10 Feature Importances')
    plt.show()

## 15. Sample Prediction

In [ ]:
sample = X_test.iloc[[0]]
print('Features:')
display(sample)
pred = best_model.predict(sample)
print(f'\nPredicted Price: ${pred[0]:.2f}')
print(f'Actual Price: ${y_test.iloc[0]}')

## 16. Conclusion
The models successfully predict car prices with Gradient Boosting achieving the best performance.